# QOMM — running the experiment interactively

**This notebook re-runs the measurement.** Where `reproduce.ipynb` only reads past results,
this one rebuilds the market and runs it on the spot.

One arm of 48,000 steps takes **0.13 s**, so "what if the attacker knew more",
"what if the market were thinner", "what if the disclosure budget were larger"
are answered while you are still thinking about them.

**It does the same thing the real runner does.** `qomm_sim/lab.py` calls the same
functions in the same order as `scripts/run_sim_matrix.py`.
If this and the artifacts disagree at the same settings, that is a bug in this notebook
and not a discovery --- `tests/test_lab.py` pins it.

In [ ]:
import sys, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from qomm_sim import lab
import matplotlib.pyplot as plt

OBLIVIOUS, PLAIN, NEUTRAL = "#1f4e79", "#c0504d", "#7f7f7f"

def show(ax, title, xlabel, ylabel):
    ax.set_title(title, fontsize=10, loc="left"); ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9); ax.grid(True, color="#dddddd", linewidth=0.6)
    ax.set_axisbelow(True); [ax.spines[s].set_visible(False) for s in ("top", "right")]

t = time.perf_counter()
setup = lab.build()                       # defaults match the paper
print(setup.describe(), f"  ({time.perf_counter()-t:.2f} s)")

## How to use it (this is all of it)

```python
setup = lab.build(steps=48_000, n_mm=16, tape=None)   # build one market
row   = lab.arm(setup, protocol="qomm_rfq", rho=0.5)  # run one arm
rows  = lab.sweep(setup, "rho", [0, 0.25, 0.5, 1.0])  # sweep one knob
rows  = lab.compare(setup, ("qomm_rfq", "plain_rfq")) # one market, several protocols
print(lab.table(rows))
```

**The market is not rebuilt between arms.** Rebuilding would give each protocol a
different order flow, mixing the market's variation into the comparison.

In [ ]:
print(lab.table(lab.compare(setup)))

## Q1. What changes with how much wallet ownership the attacker knows

In the ordinary scheme the venue broadcasts the request, but what appears is an address, not a name.
To say "firm A asked", the attacker has to know whose address that is, from outside.
`rho` is that fraction.

**Sweep the knob.** Only one of the two lines moves.

In [ ]:
rhos = [0.0, 0.05, 0.12, 0.25, 0.5, 0.75, 1.0]
fig, ax = plt.subplots(figsize=(5.2, 3.4))
for protocol, colour, label in (("qomm_rfq", OBLIVIOUS, "this design"),
                                ("plain_rfq", PLAIN, "plain RFQ")):
    rows = lab.sweep(setup, "rho", rhos, protocol=protocol)
    ax.plot(rhos, [r["detection_auc"] for r in rows], "o-", color=colour,
            label=label, markersize=4)
ax.axhline(0.5, color=NEUTRAL, linestyle=":", linewidth=1)
show(ax, "prior knowledge against how well it can be guessed", "fraction of wallets whose owner is known", "AUC")
ax.legend(fontsize=8, frameon=False); plt.show()

The blue line is flat because this design hands the attacker an empty clue.
Every candidate ties, and the measure comes out at exactly 0.5 by arithmetic.

**So how large the gap is depends on what is assumed about the attacker.** What is
about the protocol is the independence. Around 0.05 the ordinary scheme guesses little either.

In [ ]:
# Move the knob directly, where ipywidgets is available
try:
    from ipywidgets import interact, FloatSlider
    @interact(rho=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.5, description="rho"))
    def _(rho):
        print(lab.table(lab.compare(setup, rho=rho)))
except ImportError:
    print("no ipywidgets; edit rhos in the cell above instead")

## Q2. Making the market thicker or thinner

Change the arrival rate. **Making it thicker removes the very thing this design protects**:
every entity settles somewhere every time, so the situation of having asked and
settled nothing stops occurring.
Watch `detection_cells`, the number of cases that could be scored.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(9.2, 3.4))
rates = [0.02, 0.05, 0.10, 0.15, 0.30, 0.60]
aucs, cells = [], []
for rate in rates:
    s = lab.build(arrival_rate=rate)
    row = lab.arm(s, protocol="plain_rfq")
    aucs.append(row["detection_auc"]); cells.append(row["detection_cells"])
left.plot([r * 20 for r in rates], aucs, "o-", color=PLAIN, markersize=4)
show(left, "plain RFQ, how well it can be guessed", "orders per second", "AUC")
right.plot([r * 20 for r in rates], cells, "o-", color=NEUTRAL, markersize=4)
show(right, "cases that could be scored", "orders per second", "count")
right.set_yscale("log"); plt.show()
print("arrival rate:", [f"{r*20:.1f}/s" for r in rates])
print("scored:      ", cells)

## Q3. Changing the disclosure budget eps

A larger budget means less noise and a more accurate disclosure, but the budget
for continued disclosure runs out sooner. Shown on the reactive layer (`reactive=True`).

In [ ]:
eps = [0.25, 0.5, 1.0, 2.0, 4.0]
rows = lab.sweep(setup, "epsilon", eps, protocol="plain_rfq",
                 disclosure="C_dp", reactive=True)
base = lab.arm(setup, protocol="plain_rfq", disclosure="A_none", reactive=True)

fig, (left, right) = plt.subplots(1, 2, figsize=(9.2, 3.4))
left.plot(eps, [r["fill_rate"] - base["fill_rate"] for r in rows], "o-",
          color=PLAIN, markersize=4)
left.axhline(0, color=NEUTRAL, linewidth=1)
show(left, "fill rate against no disclosure", "eps per window", "difference")
right.plot(eps, [r["suppression_rate"] for r in rows], "o-", color=OBLIVIOUS, markersize=4)
show(right, "windows where disclosure halted", "eps per window", "fraction")
plt.show()
print(lab.table(rows, columns=("epsilon", "fill_rate", "mm_pnl_per_fill", "suppression_rate")))

## Q4. Putting the DP defects back on purpose

The published statistic had two defects. Both are switchable, so the state it was
published in can be reproduced.

- `debias=False` --- do not correct the upward bias from the absolute value (as published)
- `signed_sensitivity=2.0` --- twice the noise actually needed (as published)

In [ ]:
variants = [
    ("as published",   dict(debias=False, signed_sensitivity=2.0)),
    ("bias only",      dict(debias=True,  signed_sensitivity=2.0)),
    ("noise only",     dict(debias=False, signed_sensitivity=1.0)),
    ("both fixed",     dict(debias=True,  signed_sensitivity=1.0)),
]
base = lab.arm(setup, protocol="plain_rfq", disclosure="A_none", reactive=True)
print(f"  {'setting':>16} {'d fill rate':>13} {'d maker P&L':>13}")
for label, knobs in variants:
    row = lab.arm(setup, protocol="plain_rfq", disclosure="C_dp",
                  reactive=True, **knobs)
    print(f"  {label:>16} {row['fill_rate']-base['fill_rate']:>+12.4f} "
          f"{row['mm_pnl_per_fill']-base['mm_pnl_per_fill']:>+12.2f}")
print("\nOne seed only; use make dp-effect (12 seeds, paired intervals) to decide whether a difference is real")

## Q5. Swapping in a different real instrument

Swap the trade tape. Arrival rates differ by 91x across instruments, which makes
this **the most honest way to vary thickness**.

In [ ]:
available = lab.tapes()
print("available:", [p.name.split("2021")[0] for p in available] or "(artifacts/tapes is empty)")

rows = []
for path in available:
    s = lab.build(steps=2_400, window_steps=60, tape=path, tape_step_ms=1_000)
    rate = len(s.requests) / 2_400
    for protocol in ("qomm_rfq", "plain_rfq"):
        row = lab.arm(s, protocol=protocol)
        row["symbol"] = path.name.split("2021")[0]
        row["per_second"] = rate
        rows.append(row)

fig, ax = plt.subplots(figsize=(5.4, 3.4))
for protocol, colour, label in (("qomm_rfq", OBLIVIOUS, "this design"),
                                ("plain_rfq", PLAIN, "plain RFQ")):
    pts = sorted((r["per_second"], r["detection_auc"]) for r in rows
                 if r["protocol"] == protocol and r["detection_auc"] is not None)
    ax.plot([p[0] for p in pts], [p[1] for p in pts], "o", color=colour,
            label=label, markersize=5)
ax.axhline(0.5, color=NEUTRAL, linestyle=":", linewidth=1)
show(ax, "by real instrument", "orders per second", "AUC")
ax.set_xscale("log"); ax.legend(fontsize=8, frameon=False); plt.show()
print(lab.table(rows, columns=("symbol", "protocol", "detection_auc", "detection_cells", "fill_rate")))

## Q6. Changing the width of a settlement rail

From here it is the settlement side, not the market. **The cryptography actually runs** (it takes a few seconds).
The cost should be proportional to the ledger's width, and it is.

In [ ]:
import statistics, time
from zk.groups import make_group
sys.path.insert(0, str(ROOT / "scripts"))
from run_defmi import one_settlement

group = make_group("ed25519")
widths = [8, 16, 24, 32, 40]
build_ms, settle_ms, packet = [], [], []
for bits in widths:
    samples = [one_settlement(group, bits) for _ in range(3)]
    build_ms.append(statistics.median(s[0] for s in samples))
    settle_ms.append(statistics.median(s[1] for s in samples))
    packet.append(samples[0][2])
    print(f"  {bits:>2} bit  build {build_ms[-1]:>6.1f}ms  settle {settle_ms[-1]:>6.1f}ms  "
          f"{packet[-1]:>7,} B")

fig, ax = plt.subplots(figsize=(5.0, 3.4))
ax.plot(widths, settle_ms, "o-", color=PLAIN, label="settle (verify)", markersize=4)
ax.plot(widths, build_ms, "s-", color=OBLIVIOUS, label="build the package", markersize=4)
show(ax, "settlement cost follows the ledger width", "balance width (bits)", "ms")
ax.legend(fontsize=8, frameon=False); plt.show()
slope = (settle_ms[-1] - settle_ms[0]) / (widths[-1] - widths[0])
print(f"\nslope {slope:.2f} ms/bit --- deciding the width at listing beats changing the cryptography")

## Q7. Can the inventory audit be broken?

At every fill it shows the inventory moved correctly and stayed inside a limit only its owner knows.
**Try to break it.** Anything that passes and should not is a finding.

In [ ]:
from zk.state_audit import StateAuditor

auditor = StateAuditor(make_group("ed25519"), ceiling=1 << 12)
limit_value, limit_blinding = 400, auditor.key.random_blinding()
limit = auditor.commit_limit(limit_value, limit_blinding)
blinding = auditor.key.random_blinding()
opening = auditor.key.commit(0, blinding)

# three honest fills
inventory, steps = 0, []
for i, filled in enumerate([50, -20, 30]):
    nb = auditor.key.random_blinding()
    step, inventory = auditor.prove_update(
        step=i, old_inventory=inventory, old_blinding=blinding, filled=filled,
        fill_blinding=auditor.key.random_blinding(), limit=limit_value,
        limit_blinding=limit_blinding, new_blinding=nb)
    steps.append(step); blinding = nb
print("honest chain:", auditor.verify_chain(opening, steps, limit))

# now try to cheat
print("skip a step:", auditor.verify_chain(opening, [steps[0], steps[2]], limit))
looser = auditor.commit_limit(limit_value * 2, auditor.key.random_blinding())
print("swap in a looser limit:", auditor.verify_chain(opening, steps, looser))
try:
    auditor.prove_update(step=0, old_inventory=390, old_blinding=blinding, filled=-50,
                         fill_blinding=auditor.key.random_blinding(), limit=limit_value,
                         limit_blinding=limit_blinding,
                         new_blinding=auditor.key.random_blinding())
    print("breach the limit: it passed --- that is a finding")
except ValueError as exc:
    print("breach the limit: refused while building the proof ---", exc)

## Write your own experiment

The knobs `lab.build` takes, and what `lab.arm` gives back.

**market**: `steps`, `window_steps`, `seed`, `n_mm`, `n_entities`, `arrival_rate`,
`tape`, `tape_kind`（`"bybit"` / `"uniswapx"`）, `tape_step_ms`, `tape_entities`

**arm**: `protocol` (6), `disclosure` (3), `epsilon`, `reactive`,
`rho`, `debias`, `signed_sensitivity`

**returned**: `detection_auc` (how well it can be guessed), `detection_cells` (cases scored),
`entities_covered`, `informed_auc`, `fill_rate`, `mm_pnl_per_fill`, `suppression_rate`,
`_result` (the raw result, for pointing other attacks at it)

The other attacks in `qomm_sim.attackers` can be run against the raw result.

```python
from qomm_sim import attackers as atk
row = lab.arm(setup, protocol="plain_rfq")
print(atk.probing_entity(row["_result"], setup.cfg, probe_budget=len(setup.probes)))
```

**A caution**: a difference from a single seed may not be a difference.
For a real comparison, sweep `seed`, or use `make dp-effect` / `make rho-sweep`,
which produce paired intervals.

In [ ]:
# from here on, whatever you like
my = lab.build(steps=48_000, n_mm=16, arrival_rate=0.15)
print(my.describe())
print(lab.table(lab.compare(my, rho=0.12)))